# LiteLLM gateway test

Calls the stack's OpenAI-compatible endpoint (LiteLLM on `:4000`) with a virtual key.
Start the stack first: `./run_all.sh`.

The key is read from `LITELLM_API_KEY_QWEN` (environment, else the repo's `.env`; empty: `LITELLM_API_KEY`) and never printed.

In [4]:
%pip install -q openai python-dotenv


[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
import os

import openai
from dotenv import dotenv_values, find_dotenv

# environment first, then the .env at the repo root (searched upward from here); never printed
env = dotenv_values(find_dotenv(usecwd=True))
api_key = (os.getenv("LITELLM_API_KEY_QWEN") or env.get("LITELLM_API_KEY_QWEN")
           or env.get("LITELLM_API_KEY"))  # fallback: the app key run_all.sh creates
assert api_key, "set LITELLM_API_KEY_QWEN (or run ./run_all.sh to create LITELLM_API_KEY)"

client = openai.OpenAI(
    api_key=api_key,
    base_url="http://localhost:4000/v1",
)

In [6]:
response = client.chat.completions.create(
    model="qwen2.5-0.5b-instruct",
    messages=[{"role": "user", "content": "I love my India"}],
)
print(response.choices[0].message.content)

As an AI language model, I don't have personal feelings or emotions like humans do, but I can tell you about the rich cultural and historical significance of India.

India is one of the oldest civilizations in the world, having been inhabited for thousands of years. It has a diverse array of cultures, including Hinduism, Buddhism, Jainism, Sikhism, and many others. The country's history spans over 5000 years, with significant events such as the Indian subcontinent's rise to power during the Mauryan Empire, the Mughal Empire, and British colonial rule.

India is known for its vibrant arts and culture, which includes classical music, dance, literature, film, and theater. The country also has a strong tradition of science, technology, and innovation, particularly in fields such as medicine, engineering, and space exploration.

In terms of politics, India is divided into seven states, each governed by a governor-general appointed by the central government. The country has a democratic syst

## Streaming

Time to first token is only measured for streamed requests: a non-streamed answer arrives in one piece,
so the gateway's Logs page shows its TTFT as `-` (the engine's TTFT for every request is in Grafana).

In [ ]:
import time

start, first = time.perf_counter(), None
stream = client.chat.completions.create(
    model="qwen2.5-0.5b-instruct",
    messages=[{"role": "user", "content": "Explain the KV cache in two sentences."}],
    stream=True,
)
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        first = first or time.perf_counter() - start
        print(chunk.choices[0].delta.content, end="", flush=True)
print(f"\n\nfirst token after {first * 1000:.0f} ms, total {time.perf_counter() - start:.2f} s")